<a href="https://colab.research.google.com/github/pskarthikk/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
# Section 1 — Ranked actions + reason codes

from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier


# =========================================================
# 1. Repository setup
# =========================================================

repo = Path("/content/flyrank-ml-internship")

if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/pskarthikk/flyrank-ml-internship.git",
            str(repo),
        ],
        check=True,
    )

sys.path.insert(0, str(repo / "scripts"))

RAW_PATH = repo / "data/raw/content_refresh_anonymized.csv"


# =========================================================
# 2. Load the Week-5 dataset
# =========================================================

df = pd.read_csv(RAW_PATH)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# Recreate the same log features used previously.
for source, target in [
    ("impressions_90d", "log_impressions_90d"),
    ("clicks_90d", "log_clicks_90d"),
    ("sessions_90d", "log_sessions_90d"),
    ("ai_sessions_90d", "log_ai_sessions_90d"),
]:
    values = pd.to_numeric(
        df[source],
        errors="coerce",
    ).fillna(0)

    df[target] = np.log1p(values)


# =========================================================
# 3. Exact Week-5 model feature set
# =========================================================

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]


# =========================================================
# 4. Build the model matrix
# =========================================================

numeric_frame = (
    df[MODEL_NUMERIC_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

categorical_frame = (
    df[MODEL_CATEGORICAL_FEATURES]
    .fillna("unknown")
    .astype(str)
)

encoded_categorical = pd.get_dummies(
    categorical_frame,
    prefix=MODEL_CATEGORICAL_FEATURES,
    dummy_na=False,
    dtype=float,
)

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_categorical.reset_index(drop=True),
    ],
    axis=1,
)

y = df["is_declining_label"].astype(int)


# =========================================================
# 5. Reproduce the Week-6 client-grouped split
# =========================================================

client_series = (
    df["client_id"]
    .fillna("unknown")
    .astype(str)
)

unique_clients = (
    client_series
    .drop_duplicates()
    .to_numpy()
)

rng = np.random.default_rng(42)

shuffled_clients = rng.permutation(
    unique_clients
)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

held_out_clients = set(
    shuffled_clients[:test_client_count]
)

test_mask = client_series.isin(
    held_out_clients
)

train_indices = np.flatnonzero(
    ~test_mask.to_numpy()
)

test_indices = np.flatnonzero(
    test_mask.to_numpy()
)


# =========================================================
# 6. Train the validated Random Forest
# =========================================================

model = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)

model.fit(
    X.iloc[train_indices],
    y.iloc[train_indices],
)


# =========================================================
# 7. Score the held-out clients
# =========================================================

test_scores = model.predict_proba(
    X.iloc[test_indices]
)[:, 1]

queue = df.iloc[test_indices].copy().reset_index(drop=True)

queue["decline_review_score"] = test_scores


# =========================================================
# 8. Create practical reason codes
# =========================================================

def build_reason_code(row):
    reasons = []

    if row["days_since_last_update"] >= 365:
        reasons.append("STALE_CONTENT")

    elif row["days_since_last_update"] >= 181:
        reasons.append("AGING_CONTENT")

    if row["avg_position"] > 10 and row["avg_position"] != 0:
        reasons.append("SEARCH_VISIBILITY_GAP")

    if row["impressions_90d"] > 0 and row["ctr"] < 1:
        reasons.append("LOW_CTR")

    if row["content_age_days"] >= 365:
        reasons.append("MATURE_PAGE")

    if row["word_count"] < 500:
        reasons.append("THIN_CONTENT_SIGNAL")

    if not reasons:
        reasons.append("MODEL_PRIORITY")

    return "|".join(reasons[:3])


queue["reason_code"] = queue.apply(
    build_reason_code,
    axis=1,
)


# =========================================================
# 9. Map reason patterns to human actions
# =========================================================

def action_from_reason(reason_code):
    reasons = set(reason_code.split("|"))

    if "STALE_CONTENT" in reasons:
        return "Refresh content and verify current facts"

    if "LOW_CTR" in reasons and "SEARCH_VISIBILITY_GAP" in reasons:
        return "Review title/snippet and search intent alignment"

    if "SEARCH_VISIBILITY_GAP" in reasons:
        return "Review search intent, structure, and on-page relevance"

    if "THIN_CONTENT_SIGNAL" in reasons:
        return "Review content depth and completeness"

    if "AGING_CONTENT" in reasons:
        return "Review freshness and update opportunities"

    return "Human review before selecting a refresh action"


queue["recommended_action"] = queue["reason_code"].apply(
    action_from_reason
)


# =========================================================
# 10. Assign an action archetype
# =========================================================

def action_archetype(reason_code):
    reasons = set(reason_code.split("|"))

    if "STALE_CONTENT" in reasons or "AGING_CONTENT" in reasons:
        return "Freshness / Refresh"

    if "LOW_CTR" in reasons:
        return "SERP / CTR Review"

    if "SEARCH_VISIBILITY_GAP" in reasons:
        return "Search Intent / Relevance"

    if "THIN_CONTENT_SIGNAL" in reasons:
        return "Content Depth"

    return "General Review"


queue["action_archetype"] = queue["reason_code"].apply(
    action_archetype
)


# =========================================================
# 11. Rank the queue
# =========================================================

queue = queue.sort_values(
    "decline_review_score",
    ascending=False,
    kind="mergesort",
).reset_index(drop=True)

queue.insert(
    0,
    "priority_rank",
    np.arange(1, len(queue) + 1),
)


# =========================================================
# 12. Display the top 20
# =========================================================

display_columns = [
    "priority_rank",
    "content_id",
    "decline_review_score",
    "reason_code",
    "action_archetype",
    "recommended_action",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

print("=" * 90)
print("RANKED CONTENT ACTION QUEUE")
print("=" * 90)

print(
    queue[display_columns]
    .head(20)
    .to_string(index=False)
)

print("\nQueue rows:", len(queue))
print("Held-out clients:", len(held_out_clients))

print("\nAction archetype counts:")
print(
    queue["action_archetype"]
    .value_counts()
)

print("\nRanked queue generation: PASS")

RANKED CONTENT ACTION QUEUE
 priority_rank           content_id  decline_review_score                   reason_code  action_archetype                               recommended_action  content_age_days  days_since_last_update  avg_position  ctr
             1 content_0cf67ec37ab8              0.768385                       LOW_CTR SERP / CTR Review   Human review before selecting a refresh action               140                       8           3.0 0.00
             2 content_6e792cf3ce56              0.763496 SEARCH_VISIBILITY_GAP|LOW_CTR SERP / CTR Review Review title/snippet and search intent alignment               148                       8          29.1 0.06
             3 content_6e17dbac0491              0.763129 SEARCH_VISIBILITY_GAP|LOW_CTR SERP / CTR Review Review title/snippet and search intent alignment               140                       8          10.6 0.00
             4 content_52b1c884e871              0.753808 SEARCH_VISIBILITY_GAP|LOW_CTR SERP / CTR Review Re

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [8]:
# Section 2 — Intended use and limits

intended_use = {
    "primary_user": "Content editors and SEO reviewers",
    "purpose": (
        "Prioritize pages for human review when deciding which content "
        "may deserve refresh or optimization attention."
    ),
    "output_type": "Ranked decision-support queue",
    "validation_basis": (
        "Client-grouped validation on held-out clients; the queue is "
        "directional prioritization rather than a guaranteed outcome."
    ),
}

limits = [
    "The score ranks pages for review; it does not prove that a page will decline.",
    "The score does not estimate the traffic or revenue impact of an edit.",
    "Reason codes are review signals, not causal explanations of model predictions.",
    "A high score does not mean that a page should automatically be rewritten.",
    "Search position, CTR, freshness, and other observed signals can change over time.",
    "Editors must check search intent, factual accuracy, business context, and content quality before acting.",
    "The queue should not be treated as a production automation or an irreversible decision.",
]

print("=" * 90)
print("INTENDED USE")
print("=" * 90)

for key, value in intended_use.items():
    print(f"{key}: {value}")

print("\n" + "=" * 90)
print("LIMITS")
print("=" * 90)

for i, limit in enumerate(limits, start=1):
    print(f"{i}. {limit}")

print("\nIntended-use and limits statement: PASS")

INTENDED USE
primary_user: Content editors and SEO reviewers
purpose: Prioritize pages for human review when deciding which content may deserve refresh or optimization attention.
output_type: Ranked decision-support queue
validation_basis: Client-grouped validation on held-out clients; the queue is directional prioritization rather than a guaranteed outcome.

LIMITS
1. The score ranks pages for review; it does not prove that a page will decline.
2. The score does not estimate the traffic or revenue impact of an edit.
3. Reason codes are review signals, not causal explanations of model predictions.
4. A high score does not mean that a page should automatically be rewritten.
5. Search position, CTR, freshness, and other observed signals can change over time.
6. Editors must check search intent, factual accuracy, business context, and content quality before acting.
7. The queue should not be treated as a production automation or an irreversible decision.

Intended-use and limits statement

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [9]:
# Section 3 — Human review + the no-go list

human_review_rules = [
    "Confirm the page matches the intended search intent before recommending a change.",
    "Check whether the information is factually accurate and still current.",
    "Review the current search result landscape before changing title, structure, or content.",
    "Check whether the observed CTR, position, freshness, or traffic signals are plausible.",
    "Review the actual page quality rather than relying only on the model score.",
    "Consider business importance and editorial priorities before selecting an action.",
    "Use the reason code as a prompt for investigation, not as proof of a problem.",
    "Record the human decision before making a substantial content change.",
]

no_go_cases = [
    "Do not automatically rewrite, publish, delete, or redirect a page.",
    "Do not automatically change factual, legal, medical, financial, or safety-sensitive content.",
    "Do not automatically change canonical tags, indexing directives, or other high-impact technical SEO settings.",
    "Do not automatically declare a page successful or unsuccessful based only on the model score.",
    "Do not automatically infer business value, revenue impact, or causal effects from the score.",
    "Do not expose client names, private queries, URLs, or other private information in the queue.",
    "Do not use the queue as an irreversible production decision.",
]

print("=" * 90)
print("HUMAN REVIEW RULES")
print("=" * 90)

for i, rule in enumerate(human_review_rules, start=1):
    print(f"{i}. {rule}")

print("\n" + "=" * 90)
print("NO-GO: SHOULD NOT BE AUTOMATED")
print("=" * 90)

for i, case in enumerate(no_go_cases, start=1):
    print(f"{i}. {case}")

print("\nHuman-review and no-go rules: PASS")

HUMAN REVIEW RULES
1. Confirm the page matches the intended search intent before recommending a change.
2. Check whether the information is factually accurate and still current.
3. Review the current search result landscape before changing title, structure, or content.
4. Check whether the observed CTR, position, freshness, or traffic signals are plausible.
5. Review the actual page quality rather than relying only on the model score.
6. Consider business importance and editorial priorities before selecting an action.
7. Use the reason code as a prompt for investigation, not as proof of a problem.
8. Record the human decision before making a substantial content change.

NO-GO: SHOULD NOT BE AUTOMATED
1. Do not automatically rewrite, publish, delete, or redirect a page.
2. Do not automatically change factual, legal, medical, financial, or safety-sensitive content.
3. Do not automatically change canonical tags, indexing directives, or other high-impact technical SEO settings.
4. Do not a

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [10]:
# Section 4 — Monitoring / retrain triggers

monitoring_triggers = {
    "ranking_quality": (
        "Re-evaluate the ranking if measured Precision@20, Precision@50, "
        "or Precision@100 falls materially below the validated reference results."
    ),
    "data_drift": (
        "Review the model if the distribution of key inputs such as CTR, "
        "average position, content age, or days since update changes substantially."
    ),
    "action_mix": (
        "Investigate if the distribution of reason codes or action archetypes "
        "changes sharply without a clear content or traffic explanation."
    ),
    "business_context": (
        "Reassess the queue when search behavior, editorial strategy, "
        "content formats, or other important business conditions change."
    ),
    "feedback_loop": (
        "Review performance when human reviewers repeatedly reject or override "
        "the recommended action for high-ranked pages."
    ),
    "retraining": (
        "Consider retraining after meaningful changes in data distributions, "
        "content strategy, or observed ranking quality, followed by fresh validation."
    ),
}

reference_metrics = {
    "Precision@20": 0.650,
    "Precision@50": 0.740,
    "Precision@100": 0.720,
}

print("=" * 90)
print("MONITORING / RETRAIN TRIGGERS")
print("=" * 90)

for key, value in monitoring_triggers.items():
    print(f"{key}: {value}")

print("\n" + "=" * 90)
print("VALIDATED REFERENCE METRICS")
print("=" * 90)

for metric, value in reference_metrics.items():
    print(f"{metric}: {value:.3f}")

print("\nMonitoring and retrain plan: PASS")

MONITORING / RETRAIN TRIGGERS
ranking_quality: Re-evaluate the ranking if measured Precision@20, Precision@50, or Precision@100 falls materially below the validated reference results.
data_drift: Review the model if the distribution of key inputs such as CTR, average position, content age, or days since update changes substantially.
action_mix: Investigate if the distribution of reason codes or action archetypes changes sharply without a clear content or traffic explanation.
business_context: Reassess the queue when search behavior, editorial strategy, content formats, or other important business conditions change.
feedback_loop: Review performance when human reviewers repeatedly reject or override the recommended action for high-ranked pages.
retraining: Consider retraining after meaningful changes in data distributions, content strategy, or observed ranking quality, followed by fresh validation.

VALIDATED REFERENCE METRICS
Precision@20: 0.650
Precision@50: 0.740
Precision@100: 0.720

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
# Section 5 — Exports for the paper

OUTPUT_DIR = repo / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUEUE_EXPORT = OUTPUT_DIR / "w07_content_action_queue.csv"

export_columns = [
    "priority_rank",
    "content_id",
    "decline_review_score",
    "reason_code",
    "action_archetype",
    "recommended_action",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

queue[export_columns].to_csv(
    QUEUE_EXPORT,
    index=False,
)

print("=" * 90)
print("EXPORTS FOR THE PAPER")
print("=" * 90)

print(f"Queue exported to: {QUEUE_EXPORT}")
print(f"Rows exported: {len(queue)}")
print(f"Columns exported: {len(export_columns)}")

print("\nExport preview:")
display(
    pd.read_csv(QUEUE_EXPORT).head(10)
)

print("\nExport file exists:", QUEUE_EXPORT.exists())

if QUEUE_EXPORT.exists():
    print("Queue export: PASS")
else:
    print("Queue export: FAIL")

EXPORTS FOR THE PAPER
Queue exported to: /content/flyrank-ml-internship/work/outputs/w07_content_action_queue.csv
Rows exported: 2325
Columns exported: 10

Export preview:


,priority_rank,content_id,decline_review_score,reason_code,action_archetype,recommended_action,content_age_days,days_since_last_update,avg_position,ctr
0,1,content_0cf67ec37ab8,0.768385,LOW_CTR,SERP / CTR Review,Human review before selecting a refresh action,140,8,3.0,0.00
1,2,content_6e792cf3ce56,0.763496,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,148,8,29.1,0.06
2,3,content_6e17dbac0491,0.763129,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,140,8,10.6,0.00
3,4,content_52b1c884e871,0.753808,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,140,8,15.1,0.15
4,5,content_575fd096bff5,0.753051,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,140,8,13.9,0.00
5,6,content_df6b110a55c3,0.750078,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,175,20,15.1,0.00
6,7,content_07fc318d9ed0,0.741518,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,125,20,12.3,0.00
7,8,content_d275a7e021d8,0.740485,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,117,20,24.3,0.03
8,9,content_83be0494c955,0.739492,SEARCH_VISIBILITY_GAP|LOW_CTR,SERP / CTR Review,Review title/snippet and search intent alignment,140,8,13.7,0.00
9,10,content_eb53dc14317a,0.739256,LOW_CTR,SERP / CTR Review,Human review before selecting a refresh action,92,20,7.6,0.00



Export file exists: True
Queue export: PASS


In [12]:
# Section 5 — Decay / refresh insight and cost/value thinking

print("=" * 90)
print("DECAY / REFRESH INSIGHT")
print("=" * 90)

freshness_summary = (
    queue.groupby("action_archetype")
    .agg(
        pages=("content_id", "count"),
        median_content_age_days=("content_age_days", "median"),
        median_days_since_update=("days_since_last_update", "median"),
        median_score=("decline_review_score", "median"),
    )
    .sort_values("pages", ascending=False)
)

display(freshness_summary)

print(
    "\nObserved insight:"
)
print(
    "The ranked queue contains both freshness-related and non-freshness-related "
    "review opportunities. Freshness signals should therefore be treated as one "
    "possible review pathway rather than as proof that updating a page will improve "
    "performance."
)

print("\n" + "=" * 90)
print("COST / VALUE THINKING")
print("=" * 90)

cost_value_rules = [
    "Start with the highest-ranked pages because editor attention is limited.",
    "Prefer lower-cost reviews such as title, snippet, intent, and freshness checks when the evidence supports them.",
    "Reserve deeper rewrites for pages where human review identifies a meaningful content-quality opportunity.",
    "Do not assume that a higher model score means a higher financial return.",
    "Use observed performance, business importance, and expected editorial effort together when deciding which pages to act on.",
]

for i, rule in enumerate(cost_value_rules, start=1):
    print(f"{i}. {rule}")

print("\nDecay/refresh and cost/value analysis: PASS")

DECAY / REFRESH INSIGHT


,pages,median_content_age_days,median_days_since_update,median_score
action_archetype,,,,
SERP / CTR Review,1812,148.0,20.0,0.547897
General Review,370,294.0,20.0,0.481615
Freshness / Refresh,72,304.0,211.0,0.360328
Search Intent / Relevance,71,280.0,20.0,0.489697



Observed insight:
The ranked queue contains both freshness-related and non-freshness-related review opportunities. Freshness signals should therefore be treated as one possible review pathway rather than as proof that updating a page will improve performance.

COST / VALUE THINKING
1. Start with the highest-ranked pages because editor attention is limited.
2. Prefer lower-cost reviews such as title, snippet, intent, and freshness checks when the evidence supports them.
3. Reserve deeper rewrites for pages where human review identifies a meaningful content-quality opportunity.
4. Do not assume that a higher model score means a higher financial return.
5. Use observed performance, business importance, and expected editorial effort together when deciding which pages to act on.

Decay/refresh and cost/value analysis: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.